In [21]:
import json
from tqdm import tqdm

with open('/home/as5606/Datasets/Cholec_text_hallucinatory_questions/hulu_med_7B_hallucinatory_accuracy.json', 'r') as f:
    data = json.load(f)

total_unanswerable = 0  # |U|
correct_abstentions = 0  # Count of "I don't know" responses

abstention_patterns = [
    "i don't know",
    "i do not know",
    "don't know",
    "cannot answer",
    "cannot determine",
    "unable to",
]


for item in tqdm(data):
    for qa in item.get('new_questions_with_matches', []):
        hulu_answer = qa['hulu_med_answer'].lower()
        
        total_unanswerable += 1
        
        # Check if model abstained (said "I don't know")
        is_abstention = any(pattern in hulu_answer for pattern in abstention_patterns)
        
        if is_abstention:
            correct_abstentions += 1

ar = correct_abstentions / total_unanswerable if total_unanswerable > 0 else 0
print(f"Abstention Rate (AR): {ar:.4f}")

100%|██████████| 7652/7652 [00:00<00:00, 459291.26it/s]

Abstention Rate (AR): 0.7189


In [18]:
import json
from tqdm import tqdm

with open('/home/as5606/Datasets/Cholec_text_hallucinatory_questions/hulu_med_14B_answerable_answers.json', 'r') as f:
    data = json.load(f)

total_answerable = len(data)  # |A|
incorrect_abstentions = 0  # Count of "I don't know" on answerable questions

abstention_patterns = [
    "i don't know",
    "i do not know",
    "don't know",
    "cannot answer",
    "cannot determine",
    "unable to",
]

abstained_items = []

for item in tqdm(data):
    hulu_answer = item['hulu_med_answer'].lower()
    
    # Check if model abstained (said "I don't know")
    is_abstention = any(pattern in hulu_answer for pattern in abstention_patterns)
    
    if is_abstention:
        incorrect_abstentions += 1
        abstained_items.append({
            'id': item['id'],
            'question': item['original_question'],
            'expected_answer': item['original_answer'],
            'model_answer': item['hulu_med_answer']
        })

# Calculate FCR (False Confidence Rate - incorrectly abstaining on answerable)
fcr = incorrect_abstentions / total_answerable if total_answerable > 0 else 0
print(f"False Confidence Rate (FCR): {fcr:.4f}")

  0%|          | 0/7652 [00:00<?, ?it/s]

100%|██████████| 7652/7652 [00:00<00:00, 1058047.54it/s]

False Confidence Rate (FCR): 0.0191


In [28]:
import json
from tqdm import tqdm

# Load the data
with open('/home/as5606/Datasets/Cholec_text_hallucinatory_questions/hulu_med_14B_hallucinatory_answers.json', 'r') as f:
    data = json.load(f)

# Initialize counters for each type
question_types = {
    'temporal_future': {'correct': 0, 'total': 0, 'failed': []},
    'false_premise': {'correct': 0, 'total': 0, 'failed': []},
    'external_knowledge': {'correct': 0, 'total': 0, 'failed': []},
    'non_perceivable_attribute': {'correct': 0, 'total': 0, 'failed': []}
}

abstention_patterns = [
    "i don't know",
    "i do not know",
    "don't know",
    "cannot answer",
    "cannot determine",
    "unable to",
]

for item in tqdm(data):
    for qa in item.get('new_questions_with_hulu_answers', []):
        q_type = qa.get('type', 'unknown')
        hulu_answer = qa['hulu_med_answer'].lower()
        
        # Only process known types
        if q_type not in question_types:
            continue
        
        question_types[q_type]['total'] += 1
        
        # Check if model correctly abstained
        is_abstained = any(pattern in hulu_answer for pattern in abstention_patterns)
        
        if is_abstained:
            question_types[q_type]['correct'] += 1
        else:
            question_types[q_type]['failed'].append({
                'id': item['id'],
                'question': qa['question'],
                'hulu_med_answer': qa['hulu_med_answer']
            })

# Print results
print(f"{'='*70}")
print(f"HALLUCINATORY QUESTIONS ACCURACY BY TYPE")
print(f"{'='*70}\n")

total_correct = 0
total_questions = 0

for q_type in ['temporal_future', 'false_premise', 'external_knowledge', 'non_perceivable_attribute']:
    cat = question_types[q_type]
    accuracy = (cat['correct'] / cat['total'] * 100) if cat['total'] > 0 else 0
    
    print(f"{q_type.upper().replace('_', ' ')}:")
    print(f"  Total: {cat['total']}")
    print(f"  Correctly abstained (said 'I don't know'): {cat['correct']}")
    print(f"  Incorrect (hallucinated): {cat['total'] - cat['correct']}")
    print(f"  Accuracy: {accuracy:.2f}%")
    print()
    
    total_correct += cat['correct']
    total_questions += cat['total']

# Overall accuracy
overall_accuracy = (total_correct / total_questions * 100) if total_questions > 0 else 0
print(f"{'='*70}")
print(f"OVERALL HALLUCINATORY ACCURACY: {overall_accuracy:.2f}%")
print(f"{'='*70}\n")

100%|██████████| 7652/7652 [00:00<00:00, 337822.37it/s]

HALLUCINATORY QUESTIONS ACCURACY BY TYPE

TEMPORAL FUTURE:
  Total: 5762
  Correctly abstained (said 'I don't know'): 1895
  Incorrect (hallucinated): 3867
  Accuracy: 32.89%

FALSE PREMISE:
  Total: 5302
  Correctly abstained (said 'I don't know'): 1001
  Incorrect (hallucinated): 4301
  Accuracy: 18.88%

EXTERNAL KNOWLEDGE:
  Total: 229
  Correctly abstained (said 'I don't know'): 209
  Incorrect (hallucinated): 20
  Accuracy: 91.27%

NON PERCEIVABLE ATTRIBUTE:
  Total: 1402
  Correctly abstained (said 'I don't know'): 1285
  Incorrect (hallucinated): 117
  Accuracy: 91.65%

OVERALL HALLUCINATORY ACCURACY: 34.58%



In [1]:
#!/usr/bin/env python3
"""
ChatGPT hallucinatory questions accuracy by type.
Analyzes performance on temporal_future, false_premise, external_knowledge,
and non_perceivable_attribute questions.
"""

import json
from tqdm import tqdm

# Load the data
with open('/home/as5606/projects/Hulu-Med/chatGPT/chatgpt_hallucinatory_answers.json', 'r') as f:
    data = json.load(f)

# Initialize counters for each type
question_types = {
    'temporal_future': {'correct': 0, 'total': 0, 'failed': []},
    'false_premise': {'correct': 0, 'total': 0, 'failed': []},
    'external_knowledge': {'correct': 0, 'total': 0, 'failed': []},
    'non_perceivable_attribute': {'correct': 0, 'total': 0, 'failed': []}
}

abstention_patterns = [
    "i don't know",
    "i do not know",
    "don't know",
    "cannot answer",
    "cannot determine",
    "unable to",
]

for item in tqdm(data):
    for qa in item.get('new_questions_with_chatgpt_answers', []):
        q_type = qa.get('type', 'unknown')
        chatgpt_answer = qa['chatgpt_answer'].lower()
        
        # Only process known types
        if q_type not in question_types:
            continue
        
        question_types[q_type]['total'] += 1
        
        # Check if model correctly abstained
        is_abstained = any(pattern in chatgpt_answer for pattern in abstention_patterns)
        
        if is_abstained:
            question_types[q_type]['correct'] += 1
        else:
            question_types[q_type]['failed'].append({
                'id': item['id'],
                'question': qa['question'],
                'chatgpt_answer': qa['chatgpt_answer']
            })

# Print results
print(f"{'='*70}")
print(f"CHATGPT HALLUCINATORY QUESTIONS ACCURACY BY TYPE")
print(f"{'='*70}\n")

total_correct = 0
total_questions = 0

for q_type in ['temporal_future', 'false_premise', 'external_knowledge', 'non_perceivable_attribute']:
    cat = question_types[q_type]
    accuracy = (cat['correct'] / cat['total'] * 100) if cat['total'] > 0 else 0
    
    print(f"{q_type.upper().replace('_', ' ')}:")
    print(f"  Total: {cat['total']}")
    print(f"  Correctly abstained (said 'I don't know'): {cat['correct']}")
    print(f"  Incorrect (hallucinated): {cat['total'] - cat['correct']}")
    print(f"  Accuracy: {accuracy:.2f}%")
    print()
    
    total_correct += cat['correct']
    total_questions += cat['total']

# Overall accuracy
overall_accuracy = (total_correct / total_questions * 100) if total_questions > 0 else 0
print(f"{'='*70}")
print(f"OVERALL HALLUCINATORY ACCURACY: {overall_accuracy:.2f}%")
print(f"{'='*70}\n")


100%|██████████| 7652/7652 [00:00<00:00, 453572.84it/s]

CHATGPT HALLUCINATORY QUESTIONS ACCURACY BY TYPE

TEMPORAL FUTURE:
  Total: 5762
  Correctly abstained (said 'I don't know'): 5666
  Incorrect (hallucinated): 96
  Accuracy: 98.33%

FALSE PREMISE:
  Total: 5302
  Correctly abstained (said 'I don't know'): 4734
  Incorrect (hallucinated): 568
  Accuracy: 89.29%

EXTERNAL KNOWLEDGE:
  Total: 229
  Correctly abstained (said 'I don't know'): 221
  Incorrect (hallucinated): 8
  Accuracy: 96.51%

NON PERCEIVABLE ATTRIBUTE:
  Total: 1402
  Correctly abstained (said 'I don't know'): 1293
  Incorrect (hallucinated): 109
  Accuracy: 92.23%

OVERALL HALLUCINATORY ACCURACY: 93.85%



In [3]:
#!/usr/bin/env python3
"""
Gemini hallucinatory questions accuracy by type.
Analyzes performance on temporal_future, false_premise, external_knowledge,
and non_perceivable_attribute questions.
"""

import json
from tqdm import tqdm

# Load the data
with open('/home/as5606/projects/Hulu-Med/gemini/gemini_hallucinatory_answers.json', 'r') as f:
    data = json.load(f)

# Initialize counters for each type
question_types = {
    'temporal_future': {'correct': 0, 'total': 0, 'failed': []},
    'false_premise': {'correct': 0, 'total': 0, 'failed': []},
    'external_knowledge': {'correct': 0, 'total': 0, 'failed': []},
    'non_perceivable_attribute': {'correct': 0, 'total': 0, 'failed': []}
}

abstention_patterns = [
    "i don't know",
    "i do not know",
    "don't know",
    "cannot answer",
    "cannot determine",
    "unable to",
]

for item in tqdm(data):
    for qa in item.get('new_questions_with_gemini_answers', []):
        q_type = qa.get('type', 'unknown')
        gemini_answer = qa.get('gemini_answer')
        
        # Skip if answer is None
        if gemini_answer is None:
            continue
        
        gemini_answer = gemini_answer.lower()
        
        # Only process known types
        if q_type not in question_types:
            continue
        
        question_types[q_type]['total'] += 1
        
        # Check if model correctly abstained
        is_abstained = any(pattern in gemini_answer for pattern in abstention_patterns)
        
        if is_abstained:
            question_types[q_type]['correct'] += 1
        else:
            question_types[q_type]['failed'].append({
                'id': item['id'],
                'question': qa['question'],
                'gemini_answer': qa['gemini_answer']
            })

# Print results
print(f"{'='*70}")
print(f"GEMINI HALLUCINATORY QUESTIONS ACCURACY BY TYPE")
print(f"{'='*70}\n")

total_correct = 0
total_questions = 0

for q_type in ['temporal_future', 'false_premise', 'external_knowledge', 'non_perceivable_attribute']:
    cat = question_types[q_type]
    accuracy = (cat['correct'] / cat['total'] * 100) if cat['total'] > 0 else 0
    
    print(f"{q_type.upper().replace('_', ' ')}:")
    print(f"  Total: {cat['total']}")
    print(f"  Correctly abstained (said 'I don't know'): {cat['correct']}")
    print(f"  Incorrect (hallucinated): {cat['total'] - cat['correct']}")
    print(f"  Accuracy: {accuracy:.2f}%")
    print()
    
    total_correct += cat['correct']
    total_questions += cat['total']

# Overall accuracy
overall_accuracy = (total_correct / total_questions * 100) if total_questions > 0 else 0
print(f"{'='*70}")
print(f"OVERALL HALLUCINATORY ACCURACY: {overall_accuracy:.2f}%")
print(f"{'='*70}\n")


100%|██████████| 7648/7648 [00:00<00:00, 230875.24it/s]

GEMINI HALLUCINATORY QUESTIONS ACCURACY BY TYPE

TEMPORAL FUTURE:
  Total: 5757
  Correctly abstained (said 'I don't know'): 1565
  Incorrect (hallucinated): 4192
  Accuracy: 27.18%

FALSE PREMISE:
  Total: 5299
  Correctly abstained (said 'I don't know'): 1431
  Incorrect (hallucinated): 3868
  Accuracy: 27.01%

EXTERNAL KNOWLEDGE:
  Total: 229
  Correctly abstained (said 'I don't know'): 199
  Incorrect (hallucinated): 30
  Accuracy: 86.90%

NON PERCEIVABLE ATTRIBUTE:
  Total: 1401
  Correctly abstained (said 'I don't know'): 1401
  Incorrect (hallucinated): 0
  Accuracy: 100.00%

OVERALL HALLUCINATORY ACCURACY: 36.23%

